# ForgeLM — DPO Preference Alignment

Align a language model with human preferences using Direct Preference Optimization.

**Requirements:** GPU runtime required (Runtime → Change runtime type → T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/dpo_alignment.ipynb)

In [ ]:
# Step 1: Install ForgeLM
!pip install -q --no-cache-dir git+https://github.com/cemililik/ForgeLM.git
!pip uninstall -y wandb -q
!forgelm --version

In [ ]:
# Step 2: Check GPU
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Step 3: Create DPO config
# DPO requires a dataset with 'chosen' and 'rejected' columns
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-1.7B-Instruct"
  max_length: 1024
  load_in_4bit: true

lora:
  r: 16
  alpha: 32
  target_modules: ["q_proj", "v_proj"]

training:
  trainer_type: "dpo"
  dpo_beta: 0.1
  output_dir: "./dpo_checkpoints"
  num_train_epochs: 1
  per_device_train_batch_size: 2
  learning_rate: 5.0e-6

data:
  dataset_name_or_path: "argilla/ultrafeedback-binarized-preferences"
"""

with open("dpo_config.yaml", "w") as f:
    f.write(config_yaml)
print("DPO config saved!")

In [ ]:
# Step 4: Validate
!forgelm --config dpo_config.yaml --dry-run

In [ ]:
# Step 5: Train
!forgelm --config dpo_config.yaml

In [ ]:
# Step 6: Verify output
import os

model_path = "./dpo_checkpoints/final_model"
if not os.path.exists(model_path):
    print(f"Error: '{model_path}' not found. Check training output above.")
elif not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print(f"Error: adapter_config.json not found in '{model_path}'.")
else:
    print(f"Training completed! Model saved to: {model_path}")
    print(f"Files: {os.listdir(model_path)}")

## Other Alignment Methods

Change `trainer_type` in your config to switch methods:

| Method | `trainer_type` | Dataset Format | Best For |
|--------|---------------|----------------|----------|
| DPO | `"dpo"` | chosen/rejected pairs | General preference alignment |
| SimPO | `"simpo"` | chosen/rejected pairs | Lower memory (no reference model) |
| KTO | `"kto"` | completion + label (bool) | Binary feedback (production data) |
| ORPO | `"orpo"` | chosen/rejected pairs | SFT + alignment in one stage |
| GRPO | `"grpo"` | prompt only | Reasoning RL (like DeepSeek-R1) |